### **Retrieval**

#### **A typical RAG pipeline**

**RAG Pipeline**

```text
Documents
   ↓
Loading
   ↓
Chunking
   ↓
Embedding
   ↓
Vector Store
   ↓
Retriever
   ↓
LLM
   ↓
Final Answer

#### **1) What is a retriever?**

##### **A) Definition**

A retriever takes a user query and returns the most relevant documents or chunks from a knowledge source(VectorDB or VectorStore)

A retriever normally does not generate the final answer itself; it collects the relevant context required to generate the answer.

A retriever is a component that:
- Takes a user query 

- Searches the knowledge source 

- Returns the most relevant documents

A retriever’s job is to find relevant information based on the user’s question.

- User question
- Retriever searches the stored documents
- Returns the most relevant chunks 
- The LLM uses those chunks to generate the answer

**Simple example**

Suppose a vector database contains 1,000 chunks extracted from PDF documents.

The user asks: "What is semantic chunking?"

The retriever does not send all 1,000 chunks to the LLM. It searches the database and returns only the relevant chunks (context/retrieved data/ranked data):

- Chunk 12 → Definition of semantic chunking
- Chunk 48 → Example of semantic chunking
- Chunk 91 → Advantages of semantic chunking

These relevant chunks are then passed to the LLM.

**Simple analogy**

Think of a retriever as a librarian:
- User = Student
- Documents = Library books
- Retriever = Librarian
- LLM = Teacher

The student asks a question. 

The librarian finds the most relevant books or pages and gives them to the teacher. 

The teacher reads those pages and generates the final answer.

**Dummy Indicative code**

retriever = vector_store.as_retriever(
 search_type="mmr",
 search_kwargs={
 "k": 4, # Final number of documents to return
 "fetch_k": 20 # Candidate documents considered by MMR
 }
)

*Retrieve relevant documents*

documents = retriever.invoke("What is a vector database?")

*Display the retrieved content and metadata => Relevant context for the LLM to answer*

for document in documents:
 print(document.page_content)
 print(document.metadata)
 print("-" * 50)


**Parameters in a dummy code**
- k = final number of results retrieved from vectorstore
- filter = metadata restriction
- score_threshold = minimum acceptable relevance score. Referes on the lines of similarity search say for example cosine similarity between -1 to 1(full match) 
- fetch_k = number of candidates fetched before MMR. 
- lambda_mult = balance between relevance and diversity in MMR(maximum marginal retrieval). MMR is advanced type of similarity search which works on cosine similarity itself but in a diff way. 
- search_type → decides which search algorithm runs, out of three available i.e. cosine similarity, dot product, Euclidian distance. 

xyz
- similarity → returns the most similar chunks
- similarity_score_threshold → returns only the chunks that pass the threshold
- mmr → returns relevant and diverse chunks
- search_kwargs → configures the selected search algorithm

##### **B) Similarity Search Methods**

The most commonly supported similarity metrics :

**1) Cosine Similarity** 

Measures the angle/direction similarity between two vectors.
- Higher score = More similar 
- Most commonly used for text embeddings

**2) Euclidean Distance — L2**

Measures the straight-line distance between two vectors.
- Smaller distance = More similar

**3) Dot Product / Inner Product**

Multiplies corresponding vector values and adds them.

- Higher score = More similar
- With normalized vectors, dot product becomes equivalent to cosine similarity.


**Summary**

| Method | Best result |
|---|---|
| Cosine Simialirty | `Highest Score` |
| Euclidean distance | `Lowest Distance` |
| Dot Product | `Highest Score` |

##### **C) Metadata Filtering**

A retriever should not rely only on semantic similarity. It should also support structured constraints using document metadata.
Metadata helps the retriever limit the search to documents that satisfy specific conditions such as department, year, document type, year, source, user role, etc.


**Example Metadata**

metadata = {
 "department": "HR",
 "year": 2026,
 "document_type": "policy",
 "access_role": "manager"
}

*User Query*: Show the HR leave policy for 2026.

Metadata Filter

filter = {
 "department": "HR",
 "year": 2026
}

The retriever will search only the documents whose metadata matches:
department = HR
year = 2026

It will ignore documents from other departments or years, even when their content is semantically similar to the query.

**Common Types of Metadata Filters**
1) *Exact-match filters* : 
Match an exact metadata value.
{"department": "HR"}

2) *Range filters* : 
Match values within a range.
{"year": {"$gte": 2024, "$lte": 2026}}

3) *Boolean filters* : 
Combine multiple conditions using AND, OR, or NOT.
{
 "$and": [
 {"department": "HR"},
 {"year": 2026}
 ]
}
4) *Date filters* : Retrieve documents created or updated within a specific date range.

5) *Department filters* : 
Restrict retrieval to departments such as HR, Finance, Legal, or Engineering.
6) *Document-type filters* : 
Restrict retrieval to policies, reports, invoices, manuals, or contracts.
7) *Source filters* :
Search only selected PDFs, websites, databases, or repositories.
8) *Tenant filters* :
Ensure that users can retrieve documents only from their own organization or tenant.
9) *Role-based access filters* :
Restrict documents according to roles such as employee, manager, HR, or administrator.
10) *Pre-filtering and post-filtering* :
Decide whether metadata restrictions are applied before or after the retrieval operation. 

Metadata Filtering broadly can be categorised into 2 types : 

A) Pre-filtering : Filter first → Search later

B) Post-filtering : Search first → Filter later

**Pre-filtering**

It means applying metadata conditions before running vector or keyword search. 

All stored documents ==> Apply metadata filter ==> Allowed documents only ==> Similarity or keyword search ==> Final results

Example: 

Suppose the vector database contains 10,000 documents:
- HR documents = 1,000
- Finance documents = 3,000
- Engineering documents = 4,000
- Legal documents = 2,000

The user asks:

Show the HR leave policy for 2026.

The filter is:
filter = {
 "department": "HR",
 "year": 2026
}

With pre-filtering:
```
10,000 documents
 ↓
Filter department = HR and year = 2026
 ↓
Only 150 permitted documents remain
 ↓
Similarity search runs on those 150 documents
 ↓
Most relevant HR leave-policy chunks are returned
```


**Code Example**

retriever = vector_store.as_retriever(
 search_type="similarity",
 search_kwargs={
                "k": 4,
                "filter": {
                            "department": "HR",
                            "year": 2026
                            }
                }
                                    )

documents = retriever.invoke("Show the HR leave policy for 2026.")

**Advantages of Pre-filtering**
- Searches a smaller document set
- Reduces irrelevant results
- Improves security
- Supports tenant isolation
- Prevents unauthorized documents from entering the candidate list
- Can improve retrieval speed 

**Post-filtering**

Post-filtering means running retrieval first and applying metadata conditions afterward.

```
All stored documents
 ↓
Similarity or keyword search
 ↓
Top candidate documents
 ↓
Apply metadata filter
 ↓
Final allowed results
```

**Example**

Suppose the retriever first returns the top five semantically similar documents:

Result 1 → Finance leave policy, 2026

Result 2 → HR leave policy, 2025

Result 3 → Legal leave guideline, 2026

Result 4 → HR leave policy, 2026

Result 5 → Engineering leave policy, 2026

Now the filter is applied:

filter = {
 "department": "HR",
 "year": 2026
}

After post-filtering, only one result remains:

Result 4 → HR leave policy, 2026

The retriever originally fetched five documents, but four were removed after retrieval.

**Code Example**
documents = vector_store.similarity_search(
 "Show the HR leave policy for 2026.",
 k=5
)

filtered_documents = [
 document
 for document in documents
 if document.metadata.get("department") == "HR"
 and document.metadata.get("year") == 2026
]

**Limitations of Post-filtering**

- It may return too few final results
- Relevant permitted documents may never enter the initial top-k
- Unauthorized documents may enter the intermediate candidate set
- It is less suitable for strict access control
- A larger initial k may be required

For example:
```
Initial retrieval returns top 5
 ↓
4 results fail the filter
 ↓
Only 1 final result remains
```
Even though more valid HR documents may exist in the database, they may not have appeared in the original top five

### **2) Retrieval Types**

#### **2A) Sparse Retrieval**

Sparse retrieval searches using exact keywords and term matching. 

It is an old but reliable technique for retrieving the information. 

It doesn't use dense embedding but uses sparse vectors. 

This method is used to create Vectorless RAG. 

**Example**

Query: "employee leave policy"

Returns documents containing words such as: employee, leave, policy

Common methods:

- BM25
- TF-IDF
- Keyword Search

#### **2B) Dense Retrieval**

Dense retrieval uses embeddings to understand the semantic meaning of the query.

**Example**

Query: "How many days off can employees take?"

It can retrieve:"Employees are entitled to 20 days of annual leave."

The exact words may be different, but the meaning is similar

#### **2C) Hybrid Retrieval**

Hybrid retrieval combines sparse and dense retrieval.

Keyword/BM25 Search (Sparse)

    +

Vector Search (Dense)

    ↓

`Combined Results`

**Example**:

Query: "HR leave policy 2026"

Sparse retrieval matches exact terms such as:
HR
leave policy
2026

Dense retrieval finds semantically similar content such as:
employee annual vacation guidelines

Hybrid retrieval combines both results for better accuracy.

### **3) Query Transformation**

Query transformation improves the user’s original query before retrieval so the system can find more relevant information. It may rewrite the query, add related
terms, or break a complex query into smaller questions. 

#### **3A) Query Rewriting**

Query rewriting converts an unclear, incomplete, or conversational query into a clearer standalone search query.

**Example** :

*Original query*:

"What did he say about it?"

*Rewritten query*:

"What did the CEO say about the 2026 acquisition?"
This is especially useful in conversational RAG, where the current question depends on previous messages.

#### **3B) Query Expansion**

Query expansion adds synonyms, related terms, acronyms, spelling variations, or alternative phrases to the original query.

**Example**:

*Original query*:

"employee leave policy"

*Expanded query*:

"employee leave policy OR vacation policy OR annual leave guidelines"

**Another example:**

*Original term*:

"car"

*Expanded terms*:

"car, automobile, vehicle"

Query expansion broadens the search and helps retrieve documents that use different words for the same concept.

```
Original query
 ↓
Add related terms or synonyms
 ↓
Broader retrieval
```

#### **3C) Query Decomposition**

Query decomposition breaks a complex question into smaller and simpler sub-questions.

**Example**:

*Original query*:

"Compare the revenue of Company A and Company B in 2025 and explain why their growth rates were different."

It can be decomposed into:

Sub-query 1:
What was Company A's revenue in 2025?

Sub-query 2:
What was Company B's revenue in 2025?

Sub-query 3:
What factors affected Company A's growth?

Sub-query 4:
What factors affected Company B's growth?

The system retrieves information for each sub-query and combines the results to answer the original question. Query decomposition is useful for comparison, multi - hop, and complex questions. 

```
Complex query
 ↓
Multiple smaller sub-queries
 ↓
Retrieve evidence for each query
 ↓
Combine the results
```

### **4) Re-ranking**

Reranking is a second-stage retrieval process that takes an initial set of retrieved documents, calculates a more accurate relevance score for each document basis teh user query, and reorders arranges them from most relevant to least relevant and the most relevants ones are sent to the LLM. 

```
User Query
 ↓
Initial Retriever
(BM25, Vector Search, or Hybrid Search)
 ↓
Top Candidate Documents
 ↓
Reranker
 ↓
Reordered by Relevance
 ↓
Top Documents Sent to the LLM
```


##### **4A) Why do we need Re-ranking**

The initial retriever must search thousands or millions of documents quickly. Therefore, it normally uses a fast retrieval method such as:
- BM25
- Vector similarity search
- Hybrid retrieval

Fast retrieval provides good candidates, but their original order may not be perfectly accurate.

A reranker applies a more powerful model only to this smaller candidate set. 

This creates a practical balance:
- Initial Retrieval → Fast and broad
- Reranking → Slower but more accurate

Production search systems use this multi-stage architecture because an expensive ranking model can be applied to a small candidate set rather than the entire
document collection. 

##### **4B) Simple Example of Re-ranking**

Suppose the user asks:

How many days of annual leave do employees receive?

The initial retriever returns these candidates:
1. Remote Work Policy
2. Sick Leave Policy
3. Annual Leave Policy
4. Leave Carry-Forward Policy
5. Employee Attendance Policy

These documents are related to employees and leave, but the most useful document is not ranked first.

The reranker evaluates every candidate against the original query:

- Annual Leave Policy → 0.95
- Leave Carry-Forward Policy → 0.76
- Sick Leave Policy → 0.31
- Employee Attendance Policy → 0.19
- Remote Work Policy → 0.08

The reranker then produces a better order:
1. Annual Leave Policy
2. Leave Carry-Forward Policy
3. Sick Leave Policy
4. Employee Attendance Policy
5. Remote Work Policy

Only the highest-ranked documents are passed to the LLM. 

##### **4C) How does a Cross-encoder Re-ranker works?**

This is based on the cross-attention technique. 

A standard embedding retriever usually encodes the query and documents separately
```
Query → Query Vector
 +
Document → Document Vector
 ↓
 Similarity Score
```

Because document vectors can be created and stored in advance, this approach is fast enough to search large collections.

A cross-encoder reranker processes the query and one candidate document **together**:

```
[Query + Candidate Document]
 ↓
 Cross-Encoder
 ↓
 Relevance Score
```

This joint processing allows the model to examine detailed relationships between the words in the query and the document. 
However, every query-document pair must be processed separately, making cross-encoders more computationally expensive than first-stage vector retrieval.

For 20 candidate documents, the reranker conceptually evaluates:
```
Query + Document 1 → Score
Query + Document 2 → Score
Query + Document 3 → Score
...
Query + Document 20 → Score
```
It then sorts the documents by these scores


##### **4D) Re-ranking based Real Production Flow**

```
1. User submits a query
            ↓
2. Apply metadata and security filters
            ↓
3. Retrieve a broad candidate set (using BM25, vector, or hybrid search)
            ↓
4. Send the candidate set to a reranker
            ↓
5. Calculate query-document relevance scores
            ↓
6. Sort candidates by reranker score
            ↓
7. Apply an optional minimum-score threshold
            ↓
8. Send the best documents to the LLM
```

**For example**:
```
1,000,000 stored chunks
 ↓
Retriever selects 50 candidates
 ↓
Reranker reorders those 50 candidates
 ↓
Top 5 chunks are passed to the LLM
```


The values 50 and 5 are only examples. In production, candidate count and final result count are tuned according to accuracy, latency, model limits, token budget, and
cost. 

Elasticsearch exposes this candidate window as rank_window_size, and reranking services accept a query plus a candidate document list and return relevance -
ranked results. 

##### **4E) Was Re-ranking introduced for RAG**

No. Reranking existed in information retrieval and search systems before modern RAG. Traditional search systems already used multi-stage or cascade ranking:

```
Fast candidate generation
 ↓
More accurate ranking stages
 ↓
Final search results
```
- The 2011 work A Cascade Ranking Model for Efficient Ranked Retrieval formalized a multi-stage ranking architecture designed to balance search effectiveness and
computational efficiency.
- In 2019, Passage Re-ranking with BERT demonstrated that BERT could be adapted to score query-passage pairs and substantially improve passage-ranking
performance. This work helped popularize transformer-based semantic reranking, but it did not invent the general reranking concept.

RAG later adopted the same established idea:

##### **4F) Benefits of Re-ranking**

Reranking can:
- Improve the ordering of retrieved documents
- Remove weak candidates using a relevance threshold
- Reduce irrelevant context sent to the LLM
- Reduce unnecessary input tokens
- Improve evidence quality
- Work on results from sparse, dense, or hybrid retrieval


Official production implementations such as Elasticsearch and Cohere accept an initial candidate set and reorder it according to query relevance; Elasticsearch also
supports candidate-window size, score thresholds, filters, and chunk-level rescoring for long documents. 

##### **4G) Limitations of Re-ranking**

Reranking adds:
- Additional latency
- Additional computation
- Additional inference cost

Therefore, it should normally be applied only to a limited candidate set, not to every document in the database.

### **5) Multimodal-Retriever**

A multimodal retriever retrieves relevant information across different modalities, such as text and images. 

It uses multimodal embedding models to represent compatible modalities in a shared vector space, allowing text-to-image, image-to-text and image-to-image similarity search. 

**Example**
Text-to-Image

```
Text Query
 ↓
Text Encoder
 ↓
Shared Embedding Space
 ↓
Search Image Vectors
 ↓
Relevant Images
Image-to-Image
Image Query
 ↓
Image Encoder
 ↓
Image Embedding Space
 ↓
Search Image Vectors
 ↓
Similar Images
```